# Civic Affordability Notebook

Use this notebook to explore analytics marts from Postgres using a read-only user.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text

try:
    from dotenv import load_dotenv
except Exception:
    load_dotenv = None

env_path = Path('.') / '.env'
if load_dotenv and env_path.exists():
    load_dotenv(env_path)

database_url = os.getenv('DATABASE_URL')
if not database_url:
    raise RuntimeError('Missing DATABASE_URL. Set it in notebooks/.env or your shell env.')

print('DATABASE_URL loaded (redacted).')

In [ ]:
engine = create_engine(database_url)

with engine.connect() as conn:
    test = conn.execute(text('select current_database() as db, current_user as usr')).mappings().one()

print(f"Connected to db={test['db']} as user={test['usr']}")

In [ ]:
query = text(
    """
    select
      year,
      healthcare_share_pct_of_monthly_income,
      childcare_share_pct_of_monthly_income,
      estimated_mortgage_share_pct_of_monthly_income,
      known_expense_share_pct_of_monthly_income
    from analytics.mart_expense_share_monthly_income_annual
    where state_abbrev = 'CO'
    order by year
    """
)

with engine.connect() as conn:
    df = pd.read_sql(query, conn)

df.head()